Poisson regression
    * street data
    * day of week
    * weather
    * Holidays

This data was obtained via a Freedom of Information Act request filed with the City of Milwaukee in fall 2022, covering the post-LPR era after Milwaukee's Department of Public Works introduced automated plate recognition systems around early 2013.

will probably restrict the analysis to a geographical subsection of milwaukee like downtown, lower east side, and deer district area

Features I have
ISSUENO	ISSUEDATE	LOCATIONDESC1	VIODESCRIPTION

absolute shit ton of tickets over 9 years

Weather: Temperature, precipitation, snow (daily from Open-Meteo API) — affects parking behavior
Holidays: Federal + local holidays (Kaggle dataset or simple calendar) — changes traffic patterns

Target: tickets_issued_date_t
Predictors: day_of_week, month, temp, precip, is_holiday, zone, violation_type, lagged_counts
Deal with cyclical time features like day of week, month

Take just downtown and do daily total

Hypothesis: What is the rate parameter (λ) of the Poisson distribution for daily parking ticket counts, and what factors influence it?

simple GLM regression; The primary assumption is equidispersion, meaning the variance of the response variable equals its mean (
).

Focus more on feature selection/engineering and heatmaps



kaggle holidays dataset


In [7]:
import pandas as pd
import requests
import kagglehub 
import openpyxl


In [8]:
# Loading in the parking data, just one year
tickets_df = pd.read_excel("C:/Users/eckbergj/.vscode/Milwaukee-Ticket-Data-Analysis/data/tickets_csv/2014_parking_citations_issued_open_records_request.xlsx")

tickets_df.head(10)

,ISSUENO,ISSUEDATE,LOCATIONDESC1,VIODESCRIPTION
0,474576465,2014-01-01,2444 PALMER ST,UNREGISTERED/ IMPROPERLY REGISTERED VEHICLE
1,474311040,2014-01-01,5419 GALENA ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
2,474311062,2014-01-01,2371 56TH ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
3,474353456,2014-01-01,1112 PLEASANT ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
4,474365522,2014-01-01,2969 46TH ST,PARKING PROHIBITED BY OFFICIAL SIGN
5,474365533,2014-01-01,3422 49TH ST,PARKED IN CROSSWALK
6,474311106,2014-01-01,2648 56TH ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
7,474417915,2014-01-01,2151 30TH ST,TOW-AWAY ZONE (BLOCKING DRIVEWAY)
8,474455380,2014-01-01,523 34TH ST,PARKING IN EXCESS OF 24 HOURS
9,474468400,2014-01-01,833 HAWLEY RD,PARKED POSTED PRIVATE PROPERTY


In [11]:
# load in holiday data (https://www.kaggle.com/datasets/donnetew/us-holiday-dates-2004-2021)

path = kagglehub.dataset_download("donnetew/us-holiday-dates-2004-2021")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\eckbergj\.cache\kagglehub\datasets\donnetew\us-holiday-dates-2004-2021\versions\1


In [13]:
holidays_df = pd.read_csv("C:/Users/eckbergj/.vscode/Milwaukee-Ticket-Data-Analysis/data/US Holiday Dates (2004-2021).csv")

holidays_df.head(10)

,Date,Holiday,WeekDay,Month,Day,Year
0,2004-07-04,4th of July,Sunday,7,4,2004
1,2005-07-04,4th of July,Monday,7,4,2005
2,2006-07-04,4th of July,Tuesday,7,4,2006
3,2007-07-04,4th of July,Wednesday,7,4,2007
4,2008-07-04,4th of July,Friday,7,4,2008
5,2009-07-04,4th of July,Saturday,7,4,2009
6,2010-07-04,4th of July,Sunday,7,4,2010
7,2011-07-04,4th of July,Monday,7,4,2011
8,2012-07-04,4th of July,Wednesday,7,4,2012
9,2013-07-04,4th of July,Thursday,7,4,2013


In [17]:
# weather data (https://open-meteo.com/)
test_data = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=43.0389&longitude=-87.9065&start_date=2014-01-01&end_date=2014-12-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America%2FChicago")
test_data_json = test_data.json()

# minor cleaning of the weather data to get it into a dataframe
weather_df = pd.DataFrame(test_data_json['daily'])
weather_df['time'] = pd.to_datetime(weather_df['time'])
weather_df = weather_df.rename(columns={'time': 'date', 'temperature_2m_max': 'max_temp', 'temperature_2m_min': 'min_temp', 'precipitation_sum': 'precipitation'})
weather_df.head(10)


,date,max_temp,min_temp,precipitation
0,2014-01-01,-7.6,-13.3,1.3
1,2014-01-02,-8.2,-13.7,0.7
2,2014-01-03,-6.6,-17.2,0.0
3,2014-01-04,0.3,-6.1,0.7
4,2014-01-05,-4.3,-16.6,1.5
5,2014-01-06,-17.6,-25.0,0.0
6,2014-01-07,-15.9,-24.4,0.0
7,2014-01-08,-12.0,-17.4,0.0
8,2014-01-09,-3.9,-16.5,0.0
9,2014-01-10,3.1,-3.8,13.4
